In [1]:
import gc

import numpy as np
import jax
from jax import numpy as jnp
import equinox as eqx
import optax
from tqdm import tqdm

from functools import partial

from jepax.data import build_dataloader
from jepax.model import get_ijepa_model, IJEPAEncoder

key = jax.random.key(0)

encoder = IJEPAEncoder(
    key=key,
    num_channels=3,
    patch_size=2,
    dim=12,
    num_head=3,
    num_layers=6,
    img_size=32
)

/Users/anton/source/jepax/jepax/model/transformer.py:227: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  self.pe = PositionalEncoding2D(


In [2]:
wd_schedule = optax.linear_schedule(
            init_value=0.04,
            end_value=0.4,
            transition_steps=100
        )

wd_schedule(20)


optimizer = optax.adamw(
    learning_rate=0.1, 
    weight_decay=wd_schedule
)
opt_state = optimizer.init(eqx.filter(encoder, eqx.is_inexact_array))

In [3]:
def to_bf16(x):
    if isinstance(x, np.ndarray) and np.issubdtype(x.dtype, np.floating):
        return jnp.array(x, dtype=jnp.bfloat16)
    if eqx.is_array(x) and jnp.issubdtype(x.dtype, jnp.floating):
        return x.astype(jnp.bfloat16)
    return x

enc_quant = jax.tree.map(to_bf16, encoder)


In [4]:
enc_quant.transformer



Transformer(
  blocks=[
    TransformerBlock(
      attn=Attention(
        qkv_proj=Linear(
          weight=bf16[36,12],
          bias=bf16[36],
          in_features=12,
          out_features=36,
          use_bias=True
        ),
        out_proj=Linear(
          weight=bf16[12,12],
          bias=bf16[12],
          in_features=12,
          out_features=12,
          use_bias=True
        ),
        num_head=3,
        dim=12,
        causal=False
      ),
      ff=FeedForward(
        linear1=Linear(
          weight=bf16[36,12],
          bias=bf16[36],
          in_features=12,
          out_features=36,
          use_bias=True
        ),
        linear2=Linear(
          weight=bf16[12,36],
          bias=bf16[12],
          in_features=36,
          out_features=12,
          use_bias=True
        ),
        norm=LayerNorm(
          shape=(36,),
          eps=1e-05,
          use_weight=True,
          use_bias=True,
          weight=bf16[36],
          bias=bf16[36]
   

In [5]:
dataloader, num_classes, steps_per_epoch, image_size = build_dataloader(
    dataset_name="cifar10",
    data_dir="~/data",
    batch_size=16,
    is_train=True
)

batch = next(iter(dataloader))

x = batch["image"].astype(jnp.bfloat16)




In [13]:
out = enc_quant(jax.random.key(0), x[0])
out.dtype

EinopsError:  Error while processing rearrange-reduction pattern "(h ph) (w pw) c -> (h w) (c ph pw)".
 Input tensor shape: (32, 3). Additional info: {'ph': 2, 'pw': 2}.
 Wrong shape: expected 3 dims. Received 2-dim tensor.

In [ ]:
enc_quant.transformer.pe.pe.dtype

dtype('float32')

In [ ]:
x = batch["image"].astype(jnp.bfloat16)[0]
print(f"input: {x.dtype}")

# Assuming your encoder has a patch embedding + transformer
# Trace through manually:

# After patch embed (if you have one)
# x = enc_quant.patch_embed(x)
# print(f"after patch_embed: {x.dtype}")

# After PE only
x_test = jnp.ones((256, 12), dtype=jnp.bfloat16)  # fake tokens
x_pe = enc_quant.transformer.pe(x_test)
print(f"after PE: {x_pe.dtype}")

# After one block
x_block = enc_quant.transformer.blocks[0](x_pe, key=jax.random.key(0))
print(f"after block: {x_block.dtype}")

input: bfloat16
after PE: bfloat16
after block: bfloat16


In [ ]:
x = batch["image"].astype(jnp.bfloat16)[0]
print(f"input: {x.dtype}")

x = enc_quant.embed(x)
print(f"after embed: {x.dtype}")

print(f"mask_token: {enc_quant.mask_token.dtype}")

input: bfloat16
after embed: bfloat16
mask_token: bfloat16


In [ ]:
x = batch["image"].astype(jnp.bfloat16)[0]
print(f"input: {x.dtype}")

x_emb = enc_quant.embed(x)
print(f"after embed: {x_emb.dtype}")

x_out = enc_quant.transformer(x_emb, key=jax.random.key(0), train=False)
print(f"after transformer: {x_out.dtype}")

# And the full encoder call
out_full = enc_quant(jax.random.key(0), batch["image"].astype(jnp.bfloat16)[0])
print(f"full encoder: {out_full.dtype}")

input: bfloat16
after embed: bfloat16
after transformer: bfloat16
full encoder: bfloat16
